# Monsoon Onset & Soil Moisture — Core Monsoon Zone (CMZ) Demo
### ICTS Summer School on Monsoon Dynamics & Climate, 2026

This notebook demonstrates how ERA5 surface soil moisture (0–7 cm) responds to monsoon onset over the **Core Monsoon Zone (CMZ)** of India.

**What we do:**
1. 🔧 Install dependencies and clone the analysis package from GitHub
2. ☁️ Mount Google Drive to access the IMD rainfall and threshold data
3. 📥 Load the pre-regridded ERA5 soil moisture (CDO `remapcon` → IMD 1° grid)
4. 🗺️ Define the CMZ polygon mask
5. 🌧️ Detect grid-point monsoon onset dates (IMD-based algorithm)
6. 📊 Extract ±30-day composites — **CMZ grid points only for the time-series**
7. 📈 Visualise: domain-median SM profile + rainfall, and full-domain spatial maps

> **Data you need on Google Drive** (upload once, share the folder):
> - `imd/data_2025.nc` — IMD 1° daily rainfall 2025
> - `mwset1x1.nc4` — multi-week rainfall threshold climatology
> - `era5_swvl1_2025_remapcon_imd1deg.nc` — ERA5 SM already regridded to IMD 1° grid
>
> The India shapefile is bundled inside the GitHub repo — no extra upload needed.

## 🔧 Step 1 — Install Dependencies & Clone the Package

Run this cell **once** at the start of every Colab session.

In [ ]:
import os, subprocess, sys

# ── 1a. Install CDO (Climate Data Operators) for conservative regridding ──────
print("Installing CDO …")
subprocess.run(["apt-get", "install", "-y", "-q", "cdo"], check=True)
print("CDO ✓")

# ── 1b. Install Python packages not pre-installed on Colab ───────────────────
print("Installing Python packages …")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyyaml", "gcsfs", "zarr"], check=True)
print("Python packages ✓")

# ── 1c. Clone the analysis repo (skip if already present) ────────────────────
REPO_URL = "https://github.com/rmasiwal/icts_mcdm_2026.git"
REPO_DIR = "/content/icts_mcdm_2026"

if not os.path.exists(REPO_DIR):
    print("Cloning repo …")
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned ✓")
else:
    print("Repo already present — pulling latest …")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# ── 1d. Add the repo to Python path so monsoon_onset package is importable ───
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Repo on path: {REPO_DIR} ✓")

## ☁️ Step 2 — Mount Google Drive & Set Data Paths

Upload the three required files to a folder called **`ICTS_monsoon_data`** in the root of your Google Drive:

| File | What it is |
|------|-----------|
| `imd/data_2025.nc` | IMD 1° daily rainfall 2025 |
| `mwset1x1.nc4` | Multi-week rainfall threshold climatology |
| `era5_swvl1_2025_remapcon_imd1deg.nc` | ERA5 swvl1 already regridded to IMD 1° grid |

The India shapefile is **bundled in the repo** — no extra upload needed.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── Point this to the folder where you uploaded the data ─────────────────────
DRIVE_DATA = "/content/drive/MyDrive/ICTS_monsoon_data"

# ── Verify the files are reachable ────────────────────────────────────────────
YEAR = 2025

IMD_FILE    = f"{DRIVE_DATA}/imd/data_{YEAR}.nc"
THRESH_FILE = f"{DRIVE_DATA}/mwset1x1.nc4"
SM_FILE     = f"{DRIVE_DATA}/era5_swvl1_{YEAR}_remapcon_imd1deg.nc"

for label, path in [("IMD rainfall", IMD_FILE),
                    ("Threshold",    THRESH_FILE),
                    ("ERA5 SM",      SM_FILE)]:
    status = "✓  found" if os.path.exists(path) else "✗  MISSING — check the path above"
    print(f"{label:20s}: {status}")


## 📦 Step 3 — Imports & Config

Import all libraries and load the `imd_1deg` config from the cloned repo.


In [ ]:
import yaml, importlib
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as mgridspec
from matplotlib.colors import BoundaryNorm
from matplotlib.path import Path

REPO_DIR = "/content/icts_mcdm_2026"

import monsoon_onset.detector as _det
import monsoon_onset.loader   as _ldr
import monsoon_onset.plotter  as _plt
importlib.reload(_det); importlib.reload(_ldr); importlib.reload(_plt)

from monsoon_onset.loader   import load_rainfall
from monsoon_onset.detector import detect_onset
from monsoon_onset.plotter  import get_india_outline, doy_to_date_string

%matplotlib inline
os.makedirs(f"{REPO_DIR}/output", exist_ok=True)

# ── Load config, override data paths to point at Drive ───────────────────────
with open(f"{REPO_DIR}/configs/imd_1deg.yaml") as f:
    cfg = yaml.safe_load(f)

dataset_cfg = cfg["dataset"]
onset_cfg   = cfg["onset"]

# Override paths → Drive
dataset_cfg["data_path"]       = f"{DRIVE_DATA}/imd/data_{YEAR}.nc"
dataset_cfg["threshold_file"]  = THRESH_FILE
dataset_cfg["shapefile"]       = f"{REPO_DIR}/data/shapefiles/india_shapefile.shp"
shp_path = dataset_cfg["shapefile"]

print("Config loaded — data paths point to Google Drive ✓")
print(f"  IMD rainfall : {dataset_cfg['data_path']}")
print(f"  Threshold    : {dataset_cfg['threshold_file']}")
print(f"  Shapefile    : {shp_path}")


## 📥 Step 4 — Load ERA5 Soil Moisture & IMD Rainfall

Load the CDO-remapped ERA5 `swvl1` (already on the IMD 1° grid) and the IMD daily rainfall for 2025.


In [ ]:
# ── Load remapped ERA5 soil moisture ─────────────────────────────────────────
sm_ds  = xr.open_dataset(SM_FILE)
sm_var = [v for v in sm_ds.data_vars if "swvl" in v.lower() or "soil" in v.lower()][0]
sm_imd = sm_ds[sm_var]

# Standardise dimension names → 'lat' / 'lon'
_LAT_NAMES = {"latitude", "lat", "LATITUDE", "LAT", "nav_lat"}
_LON_NAMES = {"longitude", "lon", "LONGITUDE", "LON", "nav_lon"}
rename = {}
for d in sm_imd.dims:
    if d in _LAT_NAMES and d != "lat":
        rename[d] = "lat"
    if d in _LON_NAMES and d != "lon":
        rename[d] = "lon"
if rename:
    sm_imd = sm_imd.rename(rename)
    print(f"Renamed dims: {rename}")

sm_imd["time"] = pd.to_datetime(sm_imd.time.values)
sm_imd.name = "swvl1"
sm_imd.attrs["units"] = "m3 m-3"

target_lat = sm_imd.lat.values
target_lon = sm_imd.lon.values

# ── Load IMD rainfall ─────────────────────────────────────────────────────────
rainfall_da = load_rainfall(YEAR, dataset_cfg)

print(f"SM variable     : {sm_var}")
print(f"Grid            : {len(target_lat)} lats × {len(target_lon)} lons")
print(f"Lat range       : {float(sm_imd.lat.min()):.1f} – {float(sm_imd.lat.max()):.1f} °N")
print(f"Lon range       : {float(sm_imd.lon.min()):.1f} – {float(sm_imd.lon.max()):.1f} °E")
print(f"SM value range  : {float(sm_imd.min()):.4f} – {float(sm_imd.max()):.4f}  m³/m³")
print(f"Rainfall shape  : {rainfall_da.shape}")
sm_imd


## 🗺️ Step 5 — Define the Core Monsoon Zone (CMZ) Mask

The CMZ polygon (Rajeevan et al. 2010) covers central India (~18–28°N, 69–88°E).  
Grid-cell centres that fall inside the polygon are flagged `True`.


In [ ]:
# ── CMZ polygon vertices (Rajeevan et al. 2010) ──────────────────────────────
cmz_lon = np.array([74, 85, 85, 86, 86, 87, 87, 88, 88, 88, 85, 85, 82, 82, 79, 79, 78, 78, 69, 69, 74, 74])
cmz_lat = np.array([18, 18, 19, 19, 20, 20, 21, 21, 21, 24, 24, 25, 25, 26, 26, 27, 27, 28, 28, 21, 21, 18])
cmz_path = Path(np.column_stack([cmz_lon, cmz_lat]))

# ── Build 2-D boolean mask on the IMD 1° grid ─────────────────────────────────
LON_c, LAT_c = np.meshgrid(target_lon, target_lat)
pts = np.column_stack([LON_c.ravel(), LAT_c.ravel()])
cmz_mask_2d = cmz_path.contains_points(pts).reshape(LON_c.shape)

cmz_mask_da = xr.DataArray(
    cmz_mask_2d, dims=["lat", "lon"],
    coords={"lat": target_lat, "lon": target_lon},
)

n_cmz = int(cmz_mask_2d.sum())
print(f"CMZ grid points : {n_cmz}")

# ── Quick preview map ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
ax.pcolormesh(target_lon, target_lat, cmz_mask_2d.astype(float),
              cmap="Greens", vmin=0, vmax=1, shading="nearest")
for lons_b, lats_b in get_india_outline(shapefile_path=shp_path):
    ax.plot(lons_b, lats_b, color="black", linewidth=0.7)
ax.plot(np.append(cmz_lon, cmz_lon[0]), np.append(cmz_lat, cmz_lat[0]),
        color="crimson", linewidth=2, linestyle="--", label="CMZ boundary")
ax.set_xlim(65, 100); ax.set_ylim(6, 38)
ax.set_aspect("equal", adjustable="box")
ax.set_title(f"Core Monsoon Zone mask  ({n_cmz} grid points)", fontsize=11, fontweight="bold")
ax.set_xlabel("Lon (°E)"); ax.set_ylabel("Lat (°N)")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()


## 🌧️ Step 6 — Detect Monsoon Onset Dates

Run the IMD-based onset detection on the full domain, then restrict statistics to the CMZ.


In [ ]:
thresh_ds = xr.open_dataset(dataset_cfg["threshold_file"])
thresh_da = thresh_ds[dataset_cfg.get("threshold_var", "MWmean")]

onset_da = detect_onset(
    rainfall_da=rainfall_da,
    threshold_da=thresh_da,
    year=YEAR,
    cfg=onset_cfg,
)

# DOY version (full domain)
onset_doy = xr.where(onset_da.notnull(), onset_da.dt.dayofyear.astype(float), np.nan)

# ── Apply CMZ mask ────────────────────────────────────────────────────────────
onset_cmz     = onset_da.where(cmz_mask_da)
onset_doy_cmz = xr.where(onset_cmz.notnull(), onset_cmz.dt.dayofyear.astype(float), np.nan)

valid_cmz = int(onset_doy_cmz.notnull().sum())
total_cmz = int(cmz_mask_2d.sum())
print(f"CMZ grid points with valid onset : {valid_cmz} / {total_cmz}")
print(f"CMZ earliest onset : {doy_to_date_string(float(onset_doy_cmz.min()))}")
print(f"CMZ latest   onset : {doy_to_date_string(float(onset_doy_cmz.max()))}")
mean_doy = float(onset_doy_cmz.mean())
print(f"CMZ mean     onset : {doy_to_date_string(mean_doy)}  (DOY {mean_doy:.1f})")
onset_cmz


## 📊 Step 7 — Extract ±30-day Composites Centred on Onset

Two extractions:
- **CMZ only** — used for the domain-median time-series (with IQR shading)
- **Full domain** — used for the spatial maps


In [ ]:
HALF_WIN = 30
lags     = np.arange(-HALF_WIN, HALF_WIN + 1)   # -30 … +30

sm_vals    = sm_imd.values
rain_vals  = rainfall_da.values
time_idx   = pd.DatetimeIndex(sm_imd.time.values)
rain_times = pd.DatetimeIndex(rainfall_da.time.values)

n_lat = len(target_lat)
n_lon = len(target_lon)
n_lag = len(lags)

sm_centred      = np.full((n_lag, n_lat, n_lon), np.nan)
rain_centred    = np.full((n_lag, n_lat, n_lon), np.nan)
sm_centred_full = np.full((n_lag, n_lat, n_lon), np.nan)

for i, lat_v in enumerate(target_lat):
    for j, lon_v in enumerate(target_lon):
        onset_ts = onset_da.sel(lat=lat_v, lon=lon_v).values
        if pd.isna(onset_ts):
            continue
        onset_date = pd.Timestamp(onset_ts)
        for k, lag in enumerate(lags):
            tgt = onset_date + pd.Timedelta(days=int(lag))
            if tgt in time_idx:
                sm_centred_full[k, i, j] = sm_vals[time_idx.get_loc(tgt), i, j]
            if cmz_mask_2d[i, j]:
                if tgt in time_idx:
                    sm_centred[k, i, j]   = sm_vals[time_idx.get_loc(tgt), i, j]
                if tgt in rain_times:
                    rain_centred[k, i, j] = rain_vals[rain_times.get_loc(tgt), i, j]

sm_centred_da = xr.DataArray(
    sm_centred, dims=["lag", "lat", "lon"],
    coords={"lag": lags, "lat": target_lat, "lon": target_lon},
    name="swvl1_onset_centred_cmz",
    attrs={"units": "m3 m-3", "description": f"SM ±{HALF_WIN} d around onset — CMZ only"},
)
rain_centred_da = xr.DataArray(
    rain_centred, dims=["lag", "lat", "lon"],
    coords={"lag": lags, "lat": target_lat, "lon": target_lon},
    name="rain_onset_centred_cmz",
    attrs={"units": "mm/day", "description": f"Rain ±{HALF_WIN} d around onset — CMZ only"},
)
sm_centred_full_da = xr.DataArray(
    sm_centred_full, dims=["lag", "lat", "lon"],
    coords={"lag": lags, "lat": target_lat, "lon": target_lon},
    name="swvl1_onset_centred_full",
    attrs={"units": "m3 m-3", "description": f"SM ±{HALF_WIN} d around onset — full domain"},
)

print(f"CMZ  grid points with SM at onset (lag=0): {int(np.isfinite(sm_centred[HALF_WIN]).sum())}")
print(f"Full grid points with SM at onset (lag=0): {int(np.isfinite(sm_centred_full[HALF_WIN]).sum())}")


## 📈 Step 8 — CMZ Composite Statistics

Compute domain-median (and IQR) soil moisture and rainfall across all CMZ grid points, centred on onset.


In [ ]:
sm_median = sm_centred_da.median(dim=["lat", "lon"])
sm_mean   = sm_centred_da.mean(dim=["lat", "lon"])
sm_p25    = sm_centred_da.quantile(0.25, dim=["lat", "lon"])
sm_p75    = sm_centred_da.quantile(0.75, dim=["lat", "lon"])

rain_median = rain_centred_da.median(dim=["lat", "lon"])
rain_mean   = rain_centred_da.mean(dim=["lat", "lon"])
rain_p25    = rain_centred_da.quantile(0.25, dim=["lat", "lon"])
rain_p75    = rain_centred_da.quantile(0.75, dim=["lat", "lon"])

pre_sm   = float(sm_centred_da.sel(lag=slice(-30, -5)).median())
onset_sm = float(sm_centred_da.sel(lag=0).median())
post_sm  = float(sm_centred_da.sel(lag=slice(5, 30)).median())
step     = post_sm - pre_sm

print(f"CMZ median SM  pre-onset  (lag −30 to −5) : {pre_sm:.4f}  m³/m³")
print(f"CMZ median SM  at onset   (lag = 0)        : {onset_sm:.4f}  m³/m³")
print(f"CMZ median SM  post-onset (lag +5 to +30)  : {post_sm:.4f}  m³/m³")
print(f"Step change (post − pre)                    : {step:+.4f}  m³/m³")
print(f"CMZ median rain at onset (lag=0)            : {float(rain_centred_da.sel(lag=0).median(skipna=True)):.2f}  mm/day")


## 🖼️ Step 9 — Visualise: Time-series + Spatial Maps

Three panels:
- **A** — CMZ domain-mean SM time-series (±30 days, IQR shading) + rainfall on right axis
- **B** — Spatial map of SM at onset (`lag = 0`)
- **C** — Spatial map of SM at `lag = +7` days


In [ ]:
fig = plt.figure(figsize=(6, 6), dpi=150)
gs  = mgridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.1,
                          left=0.07, right=0.96, top=0.92, bottom=0.08)

map_extent = [65, 100, 6, 38]
rain_color = "steelblue"
rain_alpha = 0.35
sm_color   = "saddlebrown"

sm_at_onset = sm_centred_full_da.sel(lag=0).values
sm_lag7     = sm_centred_full_da.sel(lag=7).values

vals_plot = np.concatenate([
    sm_at_onset[np.isfinite(sm_at_onset)].ravel(),
    sm_lag7[np.isfinite(sm_lag7)].ravel(),
])
vmin = max(0.02, np.nanpercentile(vals_plot, 2))
vmax = min(0.55, np.nanpercentile(vals_plot, 98))
n_levels  = 16
sm_bounds = np.linspace(vmin, vmax, n_levels + 1)
sm_cmap   = mcolors.ListedColormap(plt.cm.YlGnBu(np.linspace(0.1, 0.95, n_levels)))
sm_norm   = BoundaryNorm(sm_bounds, n_levels)

def _map(ax, data, title, draw_cmz=True):
    im = ax.pcolormesh(target_lon, target_lat, data,
                       cmap=sm_cmap, norm=sm_norm, shading="nearest", rasterized=True)
    for lons_b, lats_b in get_india_outline(shapefile_path=shp_path):
        ax.plot(lons_b, lats_b, color="black", linewidth=0.7)
    if draw_cmz:
        ax.plot(np.append(cmz_lon, cmz_lon[0]),
                np.append(cmz_lat, cmz_lat[0]),
                color="crimson", linewidth=1.5, linestyle="--", zorder=10, label="CMZ")
        ax.legend(fontsize=7, frameon=False, loc="lower right")
    ax.patch.set_visible(False)
    ax.set_xlim(*map_extent[:2]); ax.set_ylim(*map_extent[2:])
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xlabel("Lon (°E)", fontsize=8); ax.tick_params(labelsize=7)
    return im

# Panel A
axA = fig.add_subplot(gs[0, :])
axA.fill_between(lags, sm_p25.values, sm_p75.values, color=sm_color, alpha=0.20, label="SM IQR (25–75 %)")
axA.plot(lags, sm_mean.values, color=sm_color, linewidth=2.0, label="SM CMZ mean")
axA.axvline(0, color="crimson", linewidth=1.2, linestyle="--", label="Onset (lag = 0)")
axA.set_xlabel("Days relative to onset", fontsize=9)
axA.set_ylabel("Soil moisture (m³ m⁻³)", color=sm_color, fontsize=9)
axA.tick_params(axis="y", labelcolor=sm_color, labelsize=8)
axA.set_xlim(lags[0], lags[-1])
axA.set_title("A.  CMZ domain-mean soil moisture & rainfall centred on onset", fontsize=10, fontweight="bold")
axA.spines["top"].set_visible(False)

axA_r = axA.twinx()
axA_r.fill_between(lags, rain_p25.values, rain_p75.values, color=rain_color, alpha=rain_alpha, label="Rain IQR (25–75 %)")
axA_r.plot(lags, rain_mean.values, color=rain_color, linewidth=1.5, label="Rain CMZ mean")
axA_r.set_ylabel("Rainfall (mm day⁻¹)", color=rain_color, fontsize=9)
axA_r.tick_params(axis="y", labelcolor=rain_color, labelsize=8)
axA_r.spines["top"].set_visible(False)

h1, l1 = axA.get_legend_handles_labels()
h2, l2 = axA_r.get_legend_handles_labels()
axA.legend(h1 + h2, l1 + l2, fontsize=7.5, frameon=False, loc="upper left")

# Panels B & C
axB = fig.add_subplot(gs[1, 0])
im  = _map(axB, sm_at_onset, "B.  Soil moisture at onset (lag = 0)")
axB.set_ylabel("Lat (°N)", fontsize=8)

axC = fig.add_subplot(gs[1, 1])
_map(axC, sm_lag7, "C.  Soil moisture at lag = +7 days")

cbar = fig.colorbar(im, ax=[axB, axC], orientation="horizontal",
                    pad=0.2, fraction=0.04, aspect=40, ticks=sm_bounds[::2])
cbar.set_label("Soil moisture (m³ m⁻³)", fontsize=8)
cbar.ax.tick_params(labelsize=7)
cbar.ax.set_xticklabels([f"{v:.2f}" for v in sm_bounds[::2]], fontsize=7)

out_path = f"{REPO_DIR}/output/sm_onset_cmz_{YEAR}.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {out_path}")
